In [ ]:
import pandas as pd

df = pd.read_csv("../Datensätze/qualitaet_spritzguss.csv")
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

# Welche Spalten sollen zur Vorhersage verwendet werden
X = df[["Druck", "Temperatur"]].values
y = df["Erfolg"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 0, test_size = 0.25)

## Daten vor Skalierung

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.set_cmap("bwr")
plt.scatter(X_train[:,0], X_train[:,1], c = y_train)
plt.show()

## Skalierung der Daten
Druck und Temperatur liegen in sehr unterschiedlichen Wertebereichen. Ohne Skalierung würde die Achse mit der größeren Streuung die Abstandsberechnung dominieren.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

## Daten nach Skalierung

In [ ]:
plt.scatter(X_train[:,0], X_train[:,1], c = y_train)
plt.show()

## SVM erstellen und trainieren

In [ ]:
from sklearn.svm import SVC

#model = SVC(kernel = "linear")
model = SVC(kernel = "linear", C=0.1)
model.fit(X_train, y_train)

print(model.score(X_test, y_test))

In [ ]:
from helper import plot_classifier

# Trainings-Daten plotten
plot_classifier(model, X_train, y_train, proba = False, xlabel = "Druck", ylabel = "Temperatur")

In [ ]:
# Testdaten plotten
plot_classifier(model, X_test, y_test, proba = False, xlabel = "Druck", ylabel = "Temperatur")

## Supportvektoren und Trennebene

Die folgende Grafik zeigt:
- die **Trennebene** (durchgezogene Linie, Entscheidungsfunktion = 0),
- die **Margins** (gestrichelte Linien, Entscheidungsfunktion = ±1),
- die **Supportvektoren** (umkreiste Punkte).

Die Grafik wird als `svm_supportvektoren.svg` exportiert und zusätzlich angezeigt.

In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(8, 6))

# Trainingsdaten plotten
scatter = ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train,
                     cmap="bwr", edgecolors="k", linewidths=0.3, zorder=2)

# Gitter über den Datenbereich aufspannen
x_min, x_max = X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5
y_min, y_max = X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))

# Entscheidungsfunktion auf dem Gitter auswerten
Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Trennebene (Z = 0) und Margins (Z = -1, Z = +1) einzeichnen
ax.contour(xx, yy, Z, levels=[-1, 0, 1],
           colors="k", linestyles=["--", "-", "--"], linewidths=[1, 1.8, 1])

# Supportvektoren hervorheben
ax.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
           s=180, facecolors="none", edgecolors="limegreen",
           linewidths=1.8, label="Support Vectors", zorder=3)

ax.set_xlabel("Druck (skaliert)")
ax.set_ylabel("Temperatur (skaliert)")
ax.legend(loc="best")

plt.show()